# LeetCode 3686 — Number of Stable Subsequences bằng Python

- Link: https://leetcode.com/problems/number-of-stable-subsequences/
- Độ khó: **Hard**
- Chủ đề: **DP đếm dãy con** theo trạng thái cuối (tính chẵn lẻ + độ dài đoạn chạy cùng chẵn lẻ), máy trạng thái hữu hạn, đếm modulo

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def countStableSubsequences(self, nums: List[int]) -> int:
```

## 1. Mục tiêu học

1. Đếm dãy con (theo **chỉ số**) thoả ràng buộc cục bộ bằng DP "kết thúc bằng gì".
2. Trạng thái tối thiểu: `(tính chẵn lẻ của phần tử cuối, số phần tử cuối liên tiếp cùng tính chẵn lẻ ∈ {1, 2})` — 4 trạng thái.
3. Cập nhật **từ giá trị cũ** khi thêm phần tử (mỗi phần tử dùng tối đa một lần).
4. Nhìn DP dưới dạng **tích ma trận** để thấy cấu trúc tuyến tính.
5. Kiểm chứng bằng vét cạn `2^n`.

## 2. Đề bài & ví dụ

Dãy con **ổn định** nếu không có **ba phần tử liên tiếp** (trong dãy con) cùng tính chẵn lẻ. Đếm số dãy con ổn định khác rỗng, modulo `10^9 + 7`.

**Ví dụ 1:** `[1,3,5]` → `6` (`[1,3,5]` không ổn định).

**Ví dụ 2:** `[2,3,4,2]` → `14` (chỉ `[2,4,2]` không ổn định).

**Ràng buộc:** `n ≤ 10^5`.

In [1]:
from typing import List
from itertools import combinations
import random, time

MOD = 10**9 + 7
examples = [([1, 3, 5], 6), ([2, 3, 4, 2], 14)]
for a, exp in examples:
    print(a, "-> kỳ vọng", exp)

[1, 3, 5] -> kỳ vọng 6
[2, 3, 4, 2] -> kỳ vọng 14


## 3. Kiến thức nền

### 3.1. Chỉ tính chẵn lẻ là quan trọng

Ràng buộc chỉ nhìn `x % 2` ⇒ thay mảng bằng dãy bit `0/1`. Dãy con ổn định ⇔ trong dãy bit con, không có **ba bit giống nhau liên tiếp**.

In [2]:
def is_stable(seq):
    return not any(seq[i] % 2 == seq[i + 1] % 2 == seq[i + 2] % 2 for i in range(len(seq) - 2))

print(is_stable([1, 3, 5]), is_stable([1, 3, 2, 5]), is_stable([2, 4, 2]))

False True False


### 3.2. DP "kết thúc bằng"

Khi duyệt phần tử `x` (tính chẵn lẻ `p`), mọi dãy con mới chứa `x` ở **cuối** được tạo bằng:

- dãy con chỉ gồm `x`;
- nối `x` vào một dãy con cũ kết thúc bằng tính chẵn lẻ khác `1 − p` (đoạn chạy mới dài 1);
- nối `x` vào một dãy con cũ kết thúc bằng `p` với đoạn chạy dài **1** (thành dài 2). Không nối được vào đoạn chạy dài 2.

### 3.3. Đếm phần bù để kiểm tra trực giác

Tổng số dãy con khác rỗng là `2^n − 1`. Ở ví dụ 2 (`n = 4`), có `15` dãy con và chỉ một dãy `[2,4,2]` hỏng ⇒ `14`. Đếm phần bù giúp kiểm tra nhanh các ví dụ nhỏ, nhưng **không** dùng làm lời giải vì đếm dãy con hỏng còn khó hơn (một dãy có thể chứa nhiều bộ ba hỏng chồng lên nhau).

In [3]:
nums = [2, 3, 4, 2]; n = len(nums)
subs = [[nums[i] for i in range(n) if m >> i & 1] for m in range(1, 1 << n)]
print("tổng:", len(subs), "| hỏng:", [s for s in subs if not is_stable(s)])

tổng: 15 | hỏng: [[2, 4, 2]]


### 3.4. Vì sao chỉ cần độ dài đoạn chạy tới 2?

Dãy con ổn định có đoạn chạy cuối dài 1 hoặc 2. Khi thêm phần tử cùng tính chẵn lẻ: dài 1 → 2 (vẫn ổn định), dài 2 → 3 (hỏng, loại). Khi thêm phần tử khác tính chẵn lẻ: mọi độ dài → 1. Thông tin về các phần tử **trước** đoạn chạy cuối không còn ảnh hưởng tương lai ⇒ 4 trạng thái là đủ.

## 4. Ý tưởng nền tảng / cách nghĩ trực giác

`f[p][c]` = số dãy con ổn định (đã xét tới phần tử hiện tại) kết thúc bằng tính chẵn lẻ `p`, đoạn chạy cuối dài `c ∈ {1, 2}`. Thêm `x` có tính chẵn lẻ `p`:

- `f[p][1] += 1 + f[1−p][1] + f[1−p][2]`
- `f[p][2] += f[p][1]_cũ`

Đáp án: tổng 4 giá trị.

Ví dụ 1 `[1,3,5]` (đều lẻ): sau `1`: `f[1][1] = 1`. Sau `3`: `f[1][1] = 2`, `f[1][2] = 1`. Sau `5`: `f[1][1] = 3`, `f[1][2] = 1 + 2 = 3` ⇒ tổng `6`. ✔

## 5. Cách 1 — Brute Force

In [4]:
def brute(nums):
    n = len(nums); c = 0
    for mask in range(1, 1 << n):
        if is_stable([nums[i] for i in range(n) if mask >> i & 1]): c += 1
    return c % MOD

for a, exp in examples:
    print(brute(a), "kỳ vọng", exp)

6 kỳ vọng 6
14 kỳ vọng 14


**Nút thắt:** `2^n` dãy con.

## 6. Ý tưởng thuật toán / bước đột phá

Máy trạng thái 4 trạng thái ⇒ mỗi phần tử cập nhật `O(1)` ⇒ `O(n)`. So với Cách 2 (DP theo chỉ số cuối, `O(n²)`), ta nhận ra `g[i][c]` chỉ được dùng thông qua các **tổng** theo tính chẵn lẻ của `nums[j]` — vậy chỉ cần duy trì 4 tổng đó thay vì cả mảng `g`. Đây là kỹ thuật "gộp trạng thái theo đặc trưng" rất phổ biến khi tối ưu DP đếm dãy con. Điểm cần cẩn thận: **dùng giá trị cũ** của `f[p][1]` khi tính `f[p][2]` (nếu cập nhật `f[p][1]` trước, phần tử `x` bị dùng hai lần).

## 7. Các cách tiếp theo

- **Cách 2:** DP theo chỉ số cuối `g[i][c]` — `O(n²)`: số dãy con kết thúc **tại chỉ số `i`** với đoạn chạy dài `c`.
- **Cách 3:** máy trạng thái 4 biến — `O(n)` (lời giải nộp).
- **Cách 4:** mỗi phần tử là một **ma trận 5×5** tác động lên vector `(f00, f01, f10, f11, 1)`; đáp án = tích ma trận — minh hoạ tính tuyến tính (hữu ích khi cần truy vấn đoạn bằng segment tree).

### 7.1. Cách 2 — DP theo chỉ số cuối `O(n²)`

In [5]:
def dp_quadratic(nums):
    n = len(nums); g = [[0, 0, 0] for _ in range(n)]      # g[i][c], c = 1, 2
    for i in range(n):
        p = nums[i] % 2
        g[i][1] = 1
        for j in range(i):
            if nums[j] % 2 != p: g[i][1] += g[j][1] + g[j][2]
            else: g[i][2] += g[j][1]
        g[i][1] %= MOD; g[i][2] %= MOD
    return sum(g[i][1] + g[i][2] for i in range(n)) % MOD

for a, exp in examples:
    print(dp_quadratic(a), "kỳ vọng", exp)

6 kỳ vọng 6
14 kỳ vọng 14


### 7.2. Cách 3 — Máy trạng thái `O(n)` (lời giải chính)

In [6]:
def dp_linear(nums):
    f = [[0, 0, 0], [0, 0, 0]]                             # f[p][c], c = 1, 2
    for x in nums:
        p = x & 1; q = p ^ 1
        one_old = f[p][1]
        f[p][1] = (f[p][1] + 1 + f[q][1] + f[q][2]) % MOD
        f[p][2] = (f[p][2] + one_old) % MOD
    return (f[0][1] + f[0][2] + f[1][1] + f[1][2]) % MOD

for a, exp in examples:
    print(dp_linear(a), "kỳ vọng", exp)

6 kỳ vọng 6
14 kỳ vọng 14


### 7.3. Cách 4 — Tích ma trận

In [7]:
def element_matrix(p):
    # vector v = (f01, f02, f11, f12, 1); trả M sao cho v_mới = M · v
    idx = {(0, 1): 0, (0, 2): 1, (1, 1): 2, (1, 2): 3}
    M = [[int(i == j) for j in range(5)] for i in range(5)]
    q = 1 - p
    r1 = idx[(p, 1)]; r2 = idx[(p, 2)]
    M[r1][idx[(q, 1)]] += 1; M[r1][idx[(q, 2)]] += 1; M[r1][4] += 1      # f[p][1] += f[q][1] + f[q][2] + 1
    M[r2][r1] += 1                                                         # f[p][2] += f[p][1] cũ
    return M

def mat_vec(M, v):
    return [sum(M[i][j] * v[j] for j in range(5)) % MOD for i in range(5)]

def dp_matrix(nums):
    v = [0, 0, 0, 0, 1]
    for x in nums: v = mat_vec(element_matrix(x & 1), v)
    return sum(v[:4]) % MOD

for a, exp in examples:
    print(dp_matrix(a), "kỳ vọng", exp)

6 kỳ vọng 6
14 kỳ vọng 14


## 8. TRACE từng bước

In [8]:
def trace(nums):
    f = [[0, 0, 0], [0, 0, 0]]
    for x in nums:
        p = x & 1; q = p ^ 1; one_old = f[p][1]
        f[p][1] += 1 + f[q][1] + f[q][2]; f[p][2] += one_old
        print(f"x={x} ({'lẻ' if p else 'chẵn'}): chẵn(1,2)=({f[0][1]},{f[0][2]})  lẻ(1,2)=({f[1][1]},{f[1][2]})  tổng={f[0][1]+f[0][2]+f[1][1]+f[1][2]}")

trace([2, 3, 4, 2])

x=2 (chẵn): chẵn(1,2)=(1,0)  lẻ(1,2)=(0,0)  tổng=1
x=3 (lẻ): chẵn(1,2)=(1,0)  lẻ(1,2)=(2,0)  tổng=3
x=4 (chẵn): chẵn(1,2)=(4,1)  lẻ(1,2)=(2,0)  tổng=7
x=2 (chẵn): chẵn(1,2)=(7,5)  lẻ(1,2)=(2,0)  tổng=14


In [9]:
# Liệt kê các dãy con KHÔNG ổn định của một ví dụ để đối chiếu
a = [2, 3, 4, 2, 6]
bad = [[a[i] for i in c] for k in range(3, 6) for c in combinations(range(5), k) if not is_stable([a[i] for i in c])]
print("không ổn định:", bad)
print("tổng 2^5 - 1 =", 31, "| ổn định:", dp_linear(a), "| 31 -", len(bad), "=", 31 - len(bad))

không ổn định: [[2, 4, 2], [2, 4, 6], [2, 2, 6], [4, 2, 6], [2, 4, 2, 6], [3, 4, 2, 6], [2, 3, 4, 2, 6]]
tổng 2^5 - 1 = 31 | ổn định: 24 | 31 - 7 = 24


## 9. Code nộp LeetCode

In [10]:
class Solution:
    def countStableSubsequences(self, nums: List[int]) -> int:
        MOD = 10**9 + 7
        e1 = e2 = o1 = o2 = 0               # chẵn/lẻ, đoạn chạy cuối dài 1/2
        for x in nums:
            if x & 1:
                o1, o2 = (o1 + 1 + e1 + e2) % MOD, (o2 + o1) % MOD
            else:
                e1, e2 = (e1 + 1 + o1 + o2) % MOD, (e2 + e1) % MOD
        return (e1 + e2 + o1 + o2) % MOD

sol = Solution()
for a, exp in examples:
    got = sol.countStableSubsequences(a)
    print(a, "->", got, "✔" if got == exp else "✘")

[1, 3, 5] -> 6 ✔
[2, 3, 4, 2] -> 14 ✔


## 10. Vì sao thuật toán đúng

**Bất biến:** sau khi xét `nums[0..i]`, `f[p][c]` = số dãy con ổn định (theo chỉ số) của `nums[0..i]` kết thúc bằng tính chẵn lẻ `p` với đúng `c` phần tử cuối liên tiếp cùng tính chẵn lẻ (`c ∈ {1,2}`; `c ≥ 3` không ổn định).

Khi thêm `x = nums[i+1]` (tính chẵn lẻ `p`):

- Các dãy con **không** chứa `x`: giữ nguyên (chính là giá trị cũ).
- Các dãy con chứa `x` (ở cuối): hoặc chỉ `[x]` (`c = 1`); hoặc `S + [x]` với `S` ổn định khác rỗng. Nếu `S` kết thúc bằng `1 − p` ⇒ `c = 1`, luôn ổn định. Nếu `S` kết thúc bằng `p` với `c = 1` ⇒ `c = 2`, ổn định; với `c = 2` ⇒ ba phần tử cùng tính chẵn lẻ ⇒ loại.

Đó đúng là công thức cập nhật, với mọi `S` lấy từ giá trị **cũ**. Mỗi dãy con được đếm đúng một lần (theo phần tử cuối của nó).

Đáp án cuối cùng là tổng bốn trạng thái vì mọi dãy con ổn định khác rỗng kết thúc bằng một tính chẵn lẻ và một độ dài đoạn chạy cuối duy nhất.

## 11. Phân tích Time / Space Complexity

| Cách | Time | Space |
|---|---|---|
| 1. Vét cạn | `O(2^n · n)` | `O(n)` |
| 2. DP theo chỉ số cuối | `O(n²)` | `O(n)` |
| 3. Máy trạng thái | `O(n)` | `O(1)` |
| 4. Tích ma trận | `O(n · 25)` | `O(1)` |

## 12. Bộ test

In [11]:
METHODS = [brute, dp_quadratic, dp_linear, dp_matrix, lambda a: Solution().countStableSubsequences(a)]
for a, exp in examples:
    assert all(f(a) == exp for f in METHODS)
print("Ví dụ đề: OK")

Ví dụ đề: OK


In [12]:
# 12.2 ca biên
assert Solution().countStableSubsequences([7]) == 1
assert Solution().countStableSubsequences([2, 4]) == 3
# mảng xen kẽ vẫn có dãy con không ổn định, ví dụ [1, 1, 1] lấy ở chỉ số 0, 2, 4
assert Solution().countStableSubsequences([1, 2, 1, 2, 1]) < 31
assert Solution().countStableSubsequences([1, 2, 1, 2, 1]) == brute([1, 2, 1, 2, 1])
print("Ca biên: OK")

Ca biên: OK


In [13]:
# 12.3 ngẫu nhiên
rng = random.Random(3686)
for _ in range(1500):
    a = [rng.randint(1, 6) for _ in range(rng.randint(1, 13))]
    b = brute(a)
    assert all(f(a) == b for f in METHODS[1:]), a
print("1500 test ngẫu nhiên: OK")

1500 test ngẫu nhiên: OK


In [14]:
# 12.4 test lớn
rng = random.Random(1)
a = [rng.randint(1, 10**5) for _ in range(10**5)]
t0 = time.time(); x = Solution().countStableSubsequences(a); t1 = time.time()
print("đáp án:", x, "| thời gian:", round(t1 - t0, 3), "s | khớp ma trận:", x == dp_matrix(a))

đáp án: 872815844 | thời gian: 0.023 s | khớp ma trận: True


## 13. Các lỗi người mới hay gặp

### Lỗi 1 — Cập nhật tại chỗ, dùng `f[p][1]` mới khi tính `f[p][2]`

In [15]:
def wrong_inplace(nums):
    f = [[0, 0, 0], [0, 0, 0]]
    for x in nums:
        p = x & 1; q = p ^ 1
        f[p][1] = (f[p][1] + 1 + f[q][1] + f[q][2]) % MOD
        f[p][2] = (f[p][2] + f[p][1]) % MOD                     # SAI: f[p][1] đã chứa x
    return (f[0][1] + f[0][2] + f[1][1] + f[1][2]) % MOD

print("sai:", wrong_inplace([1, 3, 5]), "| đúng:", brute([1, 3, 5]))

sai: 9 | đúng: 6


### Lỗi 2 — Hiểu "ba phần tử liên tiếp" theo chỉ số trong **mảng gốc**

Ràng buộc nói về phần tử liên tiếp **trong dãy con**. `[2, 3, 4, 2]` có `[2, 4, 2]` (chỉ số 0, 2, 3) không ổn định dù trong mảng gốc chúng không liên tiếp.

In [16]:
def wrong_original_adjacent(nums):
    n = len(nums); c = 0
    for mask in range(1, 1 << n):
        idx = [i for i in range(n) if mask >> i & 1]
        bad = any(idx[k + 2] - idx[k] == 2 and nums[idx[k]] % 2 == nums[idx[k + 1]] % 2 == nums[idx[k + 2]] % 2
                  for k in range(len(idx) - 2))                         # SAI: chỉ xét chỉ số liền nhau
        c += not bad
    return c

print("sai:", wrong_original_adjacent([2, 3, 4, 2]), "| đúng:", brute([2, 3, 4, 2]))

sai: 15 | đúng: 14


### Lỗi 3 — Quên dãy con một phần tử (`+1`)

In [17]:
def wrong_no_single(nums):
    f = [[0, 0, 0], [0, 0, 0]]
    for x in nums:
        p = x & 1; q = p ^ 1; o = f[p][1]
        f[p][1] = (f[p][1] + f[q][1] + f[q][2]) % MOD          # SAI: thiếu +1
        f[p][2] = (f[p][2] + o) % MOD
    return sum(f[0][1:]) + sum(f[1][1:])

print("sai:", wrong_no_single([1, 3, 5]), "| đúng:", brute([1, 3, 5]))

sai: 0 | đúng: 6


### Lỗi 4 — Đếm theo giá trị phân biệt

Dãy con khác nhau theo **chỉ số**: `[2, 2]` có ba dãy con `[2]`, `[2]`, `[2,2]`.

In [18]:
print("đúng (theo chỉ số):", Solution().countStableSubsequences([2, 2]), "| theo giá trị phân biệt sẽ chỉ là 2")

đúng (theo chỉ số): 3 | theo giá trị phân biệt sẽ chỉ là 2


## 14. Mẹo làm bài nhanh & nhận dạng pattern

- "Đếm dãy con không chứa mẫu cục bộ độ dài `k`" ⇒ DP trạng thái = `k − 1` "ký hiệu" cuối (ở đây rút gọn thành tính chẵn lẻ + độ dài đoạn chạy).
- Cập nhật dãy con: mới = cũ + (nối phần tử hiện tại vào cũ) ⇒ luôn tính từ giá trị cũ.
- DP tuyến tính theo phần tử ⇒ có thể viết dưới dạng ma trận ⇒ hỗ trợ truy vấn đoạn bằng segment tree lưu tích ma trận.
- Khi ràng buộc chỉ phụ thuộc một đặc trưng của phần tử (chẵn lẻ, số dư, dấu…), thay mảng bằng dãy đặc trưng trước khi nghĩ thuật toán.

## 15. Biến thể của bài

1. **Không có `k` phần tử liên tiếp cùng tính chẵn lẻ**: trạng thái `c ∈ 1..k−1`.
2. **Ràng buộc theo số dư mod 3**: trạng thái `(dư, độ dài đoạn chạy)`.
3. **Truy vấn đoạn `[l, r]`**: segment tree lưu tích ma trận 5×5.
4. **Dãy con liên tiếp (mảng con)** thay vì dãy con: đếm theo độ dài đoạn chạy trong mảng gốc.

In [19]:
# Biến thể 1: không có k phần tử liên tiếp cùng tính chẵn lẻ
def count_no_k_run(nums, k):
    f = [[0] * k for _ in range(2)]                         # f[p][c], c = 1..k-1
    for x in nums:
        p = x & 1; q = p ^ 1; old = f[p][:]
        f[p][1] = (old[1] + 1 + sum(f[q][1:])) % MOD
        for c in range(2, k): f[p][c] = (old[c] + old[c - 1]) % MOD
    return (sum(f[0][1:]) + sum(f[1][1:])) % MOD

def brute_k(nums, k):
    n = len(nums); cnt = 0
    for mask in range(1, 1 << n):
        s = [nums[i] % 2 for i in range(n) if mask >> i & 1]
        cnt += not any(len(set(s[i:i + k])) == 1 for i in range(len(s) - k + 1))
    return cnt

rng = random.Random(15)
for _ in range(400):
    a = [rng.randint(1, 6) for _ in range(rng.randint(1, 11))]; k = rng.randint(2, 4)
    assert count_no_k_run(a, k) == brute_k(a, k)
assert all(count_no_k_run(a, 3) == brute(a) for a in [[1, 3, 5], [2, 3, 4, 2]])
print("Biến thể 1 OK")

Biến thể 1 OK


## 16. So sánh các cách làm

| Cách | Ý tưởng | Time | Khi nào dùng |
|---|---|---|---|
| 1. Vét cạn | Mọi dãy con | `O(2^n n)` | Kiểm chứng |
| 2. DP chỉ số cuối | `g[i][c]` | `O(n²)` | Dễ suy ra |
| 3. Máy trạng thái | 4 biến | `O(n)` | **Lời giải chuẩn** |
| 4. Tích ma trận | Tuyến tính | `O(25 n)` | Truy vấn đoạn |

## 17. 5 bài tập vận dụng (từ dễ đến khó)

**Bài 1 (đoán output).** `countStableSubsequences([2,4,6,8])`.

**Bài 2 (tính tay).** Trace `(e1,e2,o1,o2)` cho `[1,2,3]`.

**Bài 3 (điền TODO).** Viết `count_unstable(nums)` = số dãy con khác rỗng **không** ổn định.

**Bài 4 (tự viết lại).** Viết bản truy vấn đoạn: trả lời nhiều truy vấn `[l, r]` bằng segment tree lưu tích ma trận.

**Bài 5 (mini challenge).** Đếm dãy con ổn định có **độ dài chẵn**.

In [20]:
# Bài 3 — TODO (không gọi hàm này khi chạy notebook)
def count_unstable(nums):
    # TODO
    pass

In [21]:
# Bài 5 — TODO (không gọi hàm này khi chạy notebook)
def count_even_length_stable(nums):
    # TODO: thêm chiều tính chẵn lẻ của độ dài
    pass

## 18. Gợi ý 3 mức cho bài tập

**Bài 1.** (1) Toàn chẵn ⇒ dãy con ổn định có độ dài 1 hoặc 2. (2) `4 + C(4,2) = 4 + 6`. (3) `10`.

**Bài 2.** (1) `x=1`: `o1 = 1`. (2) `x=2`: `e1 = 1 + 1 = 2`. (3) `x=3`: `o1 = 1 + 1 + 2 = 4`, `o2 = 0 + 1 = 1` ⇒ tổng `2 + 4 + 1 = 7` (= `2^3 − 1`, mọi dãy con đều ổn định).

**Bài 3.** (1) Tổng số dãy con khác rỗng `2^n − 1`. (2) Trừ số ổn định. (3) Lấy modulo.

**Bài 4.** (1) Lá = ma trận của phần tử. (2) Nút = tích (thứ tự: phải × trái). (3) Truy vấn = tích các nút phủ đoạn, áp vào vector đầu.

**Bài 5.** (1) Trạng thái thêm `parity(len)`. (2) Thêm phần tử lật tính chẵn lẻ độ dài. (3) Cộng các trạng thái độ dài chẵn.

## 19. Đáp án tham khảo cho bài tập

In [22]:
# Bài 1, 2
assert Solution().countStableSubsequences([2, 4, 6, 8]) == 10 == brute([2, 4, 6, 8])
assert Solution().countStableSubsequences([1, 2, 3]) == 7
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [23]:
# Bài 3, 4
def count_unstable_sol(nums):
    return (pow(2, len(nums), MOD) - 1 - Solution().countStableSubsequences(nums)) % MOD

def mat_mul(A, B):
    return [[sum(A[i][k] * B[k][j] for k in range(5)) % MOD for j in range(5)] for i in range(5)]

def range_queries(nums, queries):
    n = len(nums); size = 1
    while size < n: size *= 2
    I = [[int(i == j) for j in range(5)] for i in range(5)]
    t = [I] * (2 * size)
    for i, x in enumerate(nums): t[size + i] = element_matrix(x & 1)
    for i in range(size - 1, 0, -1): t[i] = mat_mul(t[2 * i + 1], t[2 * i])      # phải × trái
    out = []
    for l, r in queries:
        left, right = I, I; l += size; r += size + 1
        while l < r:
            if l & 1: left = mat_mul(t[l], left); l += 1
            if r & 1: r -= 1; right = mat_mul(right, t[r])
            l //= 2; r //= 2
        M = mat_mul(right, left); v = mat_vec(M, [0, 0, 0, 0, 1]); out.append(sum(v[:4]) % MOD)
    return out

rng = random.Random(4)
for _ in range(100):
    a = [rng.randint(1, 6) for _ in range(rng.randint(1, 10))]
    assert count_unstable_sol(a) == (2 ** len(a) - 1 - brute(a)) % MOD
    qs = []
    for _ in range(5):
        l = rng.randrange(len(a)); qs.append((l, rng.randrange(l, len(a))))
    assert range_queries(a, qs) == [dp_linear(a[l:r + 1]) for l, r in qs]
print("Bài 3, 4 OK")

Bài 3, 4 OK


In [24]:
# Bài 5
def count_even_length_stable_sol(nums):
    f = {}                                                 # (p, c, len%2) -> số dãy
    for x in nums:
        p = x & 1; q = p ^ 1; old = dict(f)
        for L in (0, 1):
            f[(p, 1, L)] = (old.get((p, 1, L), 0) + (1 if L == 1 else 0)
                            + old.get((q, 1, L ^ 1), 0) + old.get((q, 2, L ^ 1), 0)) % MOD
            f[(p, 2, L)] = (old.get((p, 2, L), 0) + old.get((p, 1, L ^ 1), 0)) % MOD
    return sum(v for (p, c, L), v in f.items() if L == 0) % MOD

rng = random.Random(5)
for _ in range(400):
    a = [rng.randint(1, 6) for _ in range(rng.randint(1, 11))]; n = len(a)
    exp = sum(1 for mask in range(1, 1 << n) if bin(mask).count("1") % 2 == 0 and is_stable([a[i] for i in range(n) if mask >> i & 1]))
    assert count_even_length_stable_sol(a) == exp
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist tự kiểm tra trước khi submit

```python
e1 = e2 = o1 = o2 = 0
for x in nums:
    if x lẻ: o1, o2 = o1 + 1 + e1 + e2, o2 + o1      # vế phải dùng giá trị CŨ
    else:    e1, e2 = e1 + 1 + o1 + o2, e2 + e1
return (e1 + e2 + o1 + o2) % MOD
```

**Checklist:**

- [ ] "Liên tiếp" là trong **dãy con**?
- [ ] Cập nhật từ giá trị cũ (gán đồng thời)?
- [ ] Có `+1` cho dãy con một phần tử?
- [ ] `% MOD` mỗi bước?
- [ ] Đã so với vét cạn?
- [ ] Đáp án không tính dãy con rỗng?